In [ ]:
# from https://github.com/jlo031/CDSE/tree/main - good docs there.

In [ ]:
import CDSE.search_and_download as CDSE_sd

In [ ]:
pos = dict()
pos['lat'] = -74.74
pos['lon'] = -106.0
results_json = CDSE_sd.search_CDSE_catalogue("Sentinel-1", pos, "2025-09-24", "2026-01-07", product_type="GRD", expand_attributes=True)

In [ ]:
p_list = results_json['value']
for p in p_list:
    print(p['Name'])

In [ ]:
# OK Johannes' code works... mine is broken - moving forward with this.

import os
import re

def s1_relative_orbit_from_filename(filename: str) -> int:
    """
    Compute Sentinel-1 relative orbit number (1..175) from a SAFE filename or path.
    Supports S1A, S1B, S1C using ESA official formulas (Product Spec 2025).

    Parameters
    ----------
    filename : str
        Example:
        'S1A_IW_GRDH_1SDV_20250114T054056_20250114T054121_055026_069ABC_1234.SAFE'
        '/some/path/S1C_IW_SLC__1SSH_20260210T045012_20260210T045039_000123_ABCD_0001.SAFE.zip'

    Returns
    -------
    int
        Relative orbit number in [1, 175].

    Raises
    ------
    ValueError
        If platform or absolute orbit number cannot be determined.
    """

    # Strip path and extension just to be safe
    base = os.path.basename(filename).upper()

    # --- Platform detection ---
    if base.startswith("S1A_"):
        platform = "S1A"
    elif base.startswith("S1B_"):
        platform = "S1B"
    elif base.startswith("S1C_"):
        platform = "S1C"
    else:
        raise ValueError(f"Could not determine platform (S1A/S1B/S1C) from '{base}'")

    # --- Absolute orbit extraction ---
    # SAFE name pattern:
    # MMM_BB_TTTR_LLLL_YYYYMMDDThhmmss_YYYYMMDDThhmmss_OOOOOO_XXXXXX_CCCC.SAFE
    # We want the 6-digit OOOOOO field.
    m = re.search(r"_(\d{6})_", base)
    if not m:
        raise ValueError(f"Could not find 6-digit absolute orbit number in '{base}'")

    abs_orbit = int(m.group(1))

    # --- ESA offsets for relative orbit ---
    if platform == "S1A":
        offset = 73
    elif platform == "S1B":
        offset = 27
    else:  # S1C
        offset = 172

    rel_orbit = ((abs_orbit - offset) % 175) + 1
    return rel_orbit


In [ ]:
# Process this list for a particular RON.

desired_ron = 167

for p in p_list:
    name = p['Name']
    ron = s1_relative_orbit_from_filename(name)

    base = os.path.basename(name).upper()

    if "_COG" in base:
        continue      # skip printing
    
    if ron == desired_ron: 
        print(name)
        #print(ron)

In [ ]:
#alphabetical order printing...

desired_ron = 167

matches = []

for p in p_list:
    name = p["Name"]
    ron = s1_relative_orbit_from_filename(name)

    base = os.path.basename(name).upper()

    # Skip cloud-optimised geotiff SAFE wrappers
    if "_COG" in base:
        continue

    if ron == desired_ron:
        matches.append(name)

# Now sort alphabetically and print
for name in sorted(matches):
    print(name)